# MM8 lab: routing and flow control (30 min)

**What changed since MM7.** A packet now crosses several routers, and the sender and receiver have to agree on how fast to go. Two questions, each anchored in Problem Set 4:

| Question | Time | | Problem Set 4 |
|---|---|---|---|
| 1 | 15 min | Distance-vector routing: what happens when a link fails? | Problems 1–2 |
| 2 | 15 min | Flow control: what if the sender is faster than the receiver? | Problem 3 |

**What you build:** three small functions in `student/mm8_network.py`, with Copilot. The routing and flow-control engines are given.

**Rhythm:** predict → run → observe → answer and check with `check_answer("MM8-1a", your_answer)`, which only says *correct* or *not correct*.

**Rules.** Prediction first, then run. Copilot writes code in `student/` only.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, "..")   # the CommSim folder
from commsim import *
from commsim.checks import check_mm8_dv, check_mm8_flow
from student.mm8_network import dv_update, advertised_window, may_send

## Question 1: distance-vector routing (15 min)

> **Question.** How does distance-vector routing react when a link fails, and what makes it recover faster?

Every router keeps a distance vector: its best known distance and next hop to every destination. In every round, all routers send their vectors to their neighbours, and each router updates its table with the Bellman-Ford rule from Lecture 8 (Problem 1).

*Conventions:* rounds are synchronous (everyone uses the vectors from the previous round), and only rounds in which some entry changed are counted. A distance of 16 or more means unreachable, as in RIP.

**Your task.** Ask Copilot to implement `dv_update` in `student/mm8_network.py` from its specification, then run the check.

In [ ]:
check_mm8_dv(dv_update)

**1a. Problem 2 with your function.** Router C receives these vectors from B, D and E, and its links to them cost 6, 3 and 5. What is C's new routing table (distances and next hops for A to F)? Compare with your answer on paper.

In [ ]:
costs = {"B": 6, "D": 3, "E": 5}
vectors = {"B": dict(zip("ABCDEF", (5, 0, 8, 12, 6, 2))),
           "D": dict(zip("ABCDEF", (16, 12, 6, 0, 9, 10))),
           "E": dict(zip("ABCDEF", (7, 6, 3, 9, 0, 4)))}
distances, next_hops = dv_update("C", costs, vectors)
print("distance ", [distances[d] for d in "ABCDEF"])
print("next hop ", [next_hops[d] for d in "ABCDEF"])

In [ ]:
check_answer("MM8-1a", ...)   # [[distances to A..F], [next hops to A..F, "-" for C itself]]

**1b. A link fails.** The engine now runs your `dv_update` at every router of the chain **Zoya – R1 – R2 – R3 – Yoshi** (every link costs 1). Once the tables have settled, the **R3–Yoshi** link fails. Infinity is 16.

*Predict first:* how many rounds until no table changes any more?

> **Your prediction:** *(write here, before you run the next cell)*

In [ ]:
r = chain_routing(dv_update, fail=("R3", "Yoshi"), poisoned_reverse=False)
print(r)
routing_plot(r);

Look at R2 and R3 in the table: whom does each of them believe, and why does the distance only grow by a little in every round?

In [ ]:
check_answer("MM8-1b", ...)   # number of rounds

**1c. Poisoned reverse.** With poisoned reverse, a router tells a neighbour "infinity" for every destination it reaches *through* that neighbour. Switch it on and rerun. How many rounds now?

In [ ]:
r_poison = chain_routing(dv_update, fail=("R3", "Yoshi"), poisoned_reverse=True)
print(r_poison)
routing_plot(r_poison);

In [ ]:
check_answer("MM8-1c", ...)   # number of rounds

**In your own words (2 to 3 sentences):** why does plain distance-vector routing count to infinity after the failure, and why does poisoned reverse stop it on this chain? Would it always be enough?

> **Your answer:** *(write here)*

## Question 2: flow control (15 min)

> **Question.** What happens when the sender is faster than the receiver can process, and what does flow control fix?

Problem 3: the receiver's CPU runs at 1000 MIPS and must copy every 64-bit packet several times; each copy costs 10 instructions. (As in the official solution, the copies are counted one after the other, so 4 copies cost 40 instructions per packet.) Packets wait in a receiver buffer of 64 packets, including the one being processed; a packet that arrives at a full buffer is dropped.

With flow control, the receiver tells the sender its **advertised window** (the free space in its buffer) every time it finishes a packet, and the sender only sends while fewer packets than that window are in flight.

**Your task.** Ask Copilot to implement `advertised_window` and `may_send` in `student/mm8_network.py`, then run the check.

In [ ]:
check_mm8_flow(advertised_window, may_send)

**Problem 3 as given:** 4 copies per packet on a 1 Gbit/s link, no flow control. The receiver can handle 1.6 Gbit/s, so nothing is lost.

In [ ]:
r = flow_control_link(copies=4, link_gbps=1.0)
print(r)

**2a. A slower receiver.** Change Problem 3 to **8 copies per packet**, still without flow control.

*Predict first:* what is the receiver's capacity in Gbit/s, and what long-run fraction of the packets will be dropped?

> **Your prediction:** *(write here, before you run the next cell)*

In [ ]:
r8 = flow_control_link(copies=8, link_gbps=1.0)
print(r8)
flow_plot(r8);

In [ ]:
check_answer("MM8-2a", ...)   # [capacity in Gbit/s, drop fraction], e.g. [1.2, 0.05]

**2b. Turn on your flow control.** Same receiver (8 copies). How many packets are dropped now, and what is the goodput?

In [ ]:
r8_fc = flow_control_link(copies=8, link_gbps=1.0, flow_control=(advertised_window, may_send))
print(r8_fc)
flow_plot(r8_fc);

In [ ]:
check_answer("MM8-2b", ...)   # [packets dropped, goodput in Gbit/s]

**2c. The window and the round trip.** Back to 4 copies, with flow control, but now the window updates need a **10 µs round trip**, and the window is the receiver buffer of 64 packets.

*Predict first:* what goodput do you expect? Then find the **smallest window** (`buffer_packets`) that reaches 1 Gbit/s. Hint: the sender can have at most one window of packets on its way per round trip.

> **Your prediction:** *(write here, before you run the next cell)*

In [ ]:
r_rtt = flow_control_link(copies=4, rtt_us=10, buffer_packets=64, flow_control=(advertised_window, may_send))
print(r_rtt)

In [ ]:
# your sweep: try different buffer_packets (windows) and find the smallest that reaches 1 Gbit/s

In [ ]:
check_answer("MM8-2c", ...)   # [goodput in Gbit/s with a window of 64, smallest window in packets]

**In your own words (2 to 3 sentences):** what does flow control fix, and what does it cost when the round trip is long?

> **Your answer:** *(write here)*